In [ ]:
from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
#from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
import os

clone the github repo

In [21]:
!mkdir -p test_repo

A subdirectory or file -p already exists.
Error occurred while processing: -p.


In [22]:
repo_path = 'test_repo'
repo= Repo.clone_from("https://github.com/swarajjoshi10-ship-it/stateful-agentic",to_path=repo_path)

loading the data from the files

In [23]:
%pwd

'c:\\Realtime-Source-Code-Analyzer\\research'

In [24]:
loader= GenericLoader.from_filesystem(repo_path,
                        glob="**/*.*",
                        suffixes=[".py"],
                        parser=LanguageParser(language=Language.PYTHON,parser_threshold=500))

In [25]:
documents= loader.load()

In [26]:
len(documents)

20

split the documents into chunks

In [27]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, 
                                                                  chunk_size=500,
                                                                  chunk_overlap=20)

In [28]:
texts= documents_splitter.split_documents(documents)

In [29]:
len(texts)

57

download the OpenAI Embeddings

In [45]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [32]:
#embeddings = OpenAIEmbeddings(disallowed_special=())
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

c:\Users\swaraj joshi\miniconda3\envs\llmapp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\swaraj joshi\miniconda3\envs\llmapp\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\swaraj joshi\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order t

setting up chroma as vector database

In [33]:
#vectordb= Chroma.from_documents(documents=texts, embedding=embeddings,persist_directory='./data')
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory='./data'
)

Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


creating an OpenAI model wrapper

In [ ]:
#llm=ChatOpenAI()
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-chat:free"
)

In [47]:
memory=ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

In [51]:
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 8}
    ),
    memory=memory
)

Question and answer

In [49]:
question = "what is the chatbot_with_tools_build_graph function"

In [52]:
result=qa(question)

AuthenticationError: Missing Authentication header

In [ ]:
print(result['answer'])